# US Sales Data Analysis

This notebook demonstrates comprehensive data cleaning and analysis of US Sales dataset.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, split, regexp_replace, 
    sum as spark_sum, round as spark_round, avg, year, month
)

spark = SparkSession.builder \
    .appName('US Sales Analysis') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Step 1: Load Raw Data

In [ ]:
# Read raw CSV data
raw_data_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/content/sample_data/US_Sales_Datasets.csv")

print("Raw Data Schema:")
raw_data_df.printSchema()

print("\nSample Raw Data:")
raw_data_df.show(5, truncate=False)

## Step 2: Data Cleaning Pipeline

### Convert Invoice Date to proper date format

In [ ]:
# Step 1: Convert Invoice Date from dd-MM-yyyy to date type
raw_data_df1 = raw_data_df.withColumn(
    "Invoice Date", 
    to_date(col("Invoice Date"), "dd-MM-yyyy")
)

print("After date conversion:")
raw_data_df1.select("Invoice Date").show(5)

### Extract Gender and Category from Product name

In [ ]:
# Step 2: Split Product into Gender and Category
# Product format: "Men's/Women's Category"
raw_data_df2 = raw_data_df1 \
    .withColumn("Gender", split(col("Product"), "'s ").getItem(0)) \
    .withColumn("Category", split(col("Product"), "'s ").getItem(1))

print("After extracting Gender and Category:")
raw_data_df2.select("Product", "Gender", "Category").show(10, truncate=False)

### Clean Units Sold (remove commas and convert to Integer)

In [ ]:
# Step 3: Clean Units Sold - remove commas and convert to Integer
raw_data_df3 = raw_data_df2 \
    .withColumn("Units Sold", regexp_replace(col("Units Sold"), ",", "")) \
    .withColumn("Units Sold", col("Units Sold").cast("Integer"))

print("After cleaning Units Sold:")
raw_data_df3.select("Units Sold").show(5)
raw_data_df3.printSchema()

### Clean Operating Margin (remove % and convert to Integer)

In [ ]:
# Step 4: Clean Operating Margin - remove % and convert to Integer
raw_data_df4 = raw_data_df3 \
    .withColumn("Operating Margin", regexp_replace(col("Operating Margin"), "%", "")) \
    .withColumn("Operating Margin", col("Operating Margin").cast("Integer"))

print("After cleaning Operating Margin:")
raw_data_df4.select("Operating Margin").show(5)

### Calculate Total Sales

In [ ]:
# Step 5: Calculate Total Sales = Units Sold * Price per Unit
raw_data_df5 = raw_data_df4 \
    .withColumn("Total Sales", col("Units Sold") * col("Price per Unit")) \
    .withColumn("Total Sales", col("Total Sales").cast("Double"))

print("After calculating Total Sales:")
raw_data_df5.select("Units Sold", "Price per Unit", "Total Sales").show(5)

### Calculate Operating Profit

In [ ]:
# Step 6: Calculate Operating Profit = Total Sales * Operating Margin / 100
raw_data_df6 = raw_data_df5 \
    .withColumn("Operating Profit", col("Total Sales") * col("Operating Margin") / 100) \
    .withColumn("Operating Profit", col("Operating Profit").cast("Double"))

print("After calculating Operating Profit:")
raw_data_df6.select("Total Sales", "Operating Margin", "Operating Profit").show(5)

### Rename columns for SQL queries

In [ ]:
# Step 7: Rename columns (remove spaces)
raw_data_df7 = raw_data_df6 \
    .withColumnRenamed("Invoice Date", "Invoice_Date") \
    .withColumnRenamed("Total Sales", "Total_Sales") \
    .withColumnRenamed("Operating Profit", "Operating_Profit") \
    .withColumnRenamed("Units Sold", "Units_Sold")

print("Final cleaned data schema:")
raw_data_df7.printSchema()

print("\nFinal cleaned data sample:")
raw_data_df7.show(5, truncate=False)

In [ ]:
# Register as temporary view for SQL queries
raw_data_df7.createOrReplaceTempView("sales")

## Analysis Queries

### Q1: Year-wise Analysis

In [ ]:
# Year-wise aggregations
q1 = spark.sql("""
    SELECT 
        YEAR(Invoice_Date) as Year,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill,
        ROUND(SUM(Total_Sales) / SUM(Units_Sold), 2) as Avg_Selling_Price,
        SUM(Units_Sold) as Total_Units_Sold
    FROM sales
    GROUP BY YEAR(Invoice_Date)
    ORDER BY Year
""")

print("Year-wise Sales Analysis:")
q1.show()

### Q2: Month-wise Analysis

In [ ]:
# Month-wise analysis
q2 = spark.sql("""
    SELECT 
        YEAR(Invoice_Date) as Year,
        MONTH(Invoice_Date) as Month,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill
    FROM sales
    GROUP BY YEAR(Invoice_Date), MONTH(Invoice_Date)
    ORDER BY Year, Month
""")

print("Month-wise Sales Analysis:")
q2.show(20)

### Q3: Category-wise Analysis

In [ ]:
# Category-wise sales
q3 = spark.sql("""
    SELECT 
        Category,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill,
        SUM(Units_Sold) as Total_Units
    FROM sales
    GROUP BY Category
    ORDER BY Total_Sales_in_Mill DESC
""")

print("Category-wise Sales Analysis:")
q3.show()

### Q4: Gender-wise Analysis

In [ ]:
# Gender-wise sales
q4 = spark.sql("""
    SELECT 
        Gender,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill,
        SUM(Units_Sold) as Total_Units
    FROM sales
    GROUP BY Gender
    ORDER BY Total_Sales_in_Mill DESC
""")

print("Gender-wise Sales Analysis:")
q4.show()

### Q5: City-wise Analysis

In [ ]:
# Top 10 cities by sales
q5 = spark.sql("""
    SELECT 
        City,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill
    FROM sales
    GROUP BY City
    ORDER BY Total_Sales_in_Mill DESC
    LIMIT 10
""")

print("Top 10 Cities by Sales:")
q5.show()

### Q6: State-wise Analysis

In [ ]:
# State-wise sales
q6 = spark.sql("""
    SELECT 
        State,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        ROUND(SUM(Operating_Profit) / 1000000, 2) as Total_Profit_in_Mill,
        COUNT(DISTINCT City) as Num_Cities
    FROM sales
    GROUP BY State
    ORDER BY Total_Sales_in_Mill DESC
    LIMIT 15
""")

print("Top 15 States by Sales:")
q6.show()

### Q7: Combined Analysis - Gender, Category by Year

In [ ]:
# Detailed breakdown
q7 = spark.sql("""
    SELECT 
        YEAR(Invoice_Date) as Year,
        Gender,
        Category,
        ROUND(SUM(Total_Sales) / 1000000, 2) as Total_Sales_in_Mill,
        SUM(Units_Sold) as Total_Units
    FROM sales
    GROUP BY YEAR(Invoice_Date), Gender, Category
    ORDER BY Year, Total_Sales_in_Mill DESC
""")

print("Year-Gender-Category Analysis:")
q7.show(30)

## Save Cleaned Data

In [ ]:
# Save cleaned data
output_path = "/tmp/us_sales_cleaned"
!rm -rf {output_path}

raw_data_df7.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Cleaned data saved to {output_path}")

In [ ]:
# Stop Spark Session
spark.stop()